# Synthetic E-commerce Sales Analysis

This notebook demonstrates a full analysis pipeline on a synthetic e-commerce sales dataset. The aim is to showcase skills relevant for business analytics, data analysis, and program management roles. We generate a dataset resembling transactional data with customer demographics, transaction details, and labels for high-value customers. The notebook includes exploratory data analysis (EDA), visualization, and predictive modeling.


## Dataset Description

The dataset contains **1,000** synthetic transaction records. Each row represents a purchase made by a customer at an e-commerce store. The columns include:

| Column | Description |
|-------|------------|
| `customer_id` | Unique identifier for the customer |
| `age` | Customer age in years |
| `gender` | Customer gender (Male, Female, Other) |
| `region` | Geographical region (North, South, East, West) |
| `product_category` | Category of the purchased product (Electronics, Clothing, Home & Kitchen, Sports, Books) |
| `purchase_date` | Date of purchase |
| `units_purchased` | Number of units purchased |
| `unit_price` | Price per unit |
| `purchase_amount` | Total purchase amount (units \* unit_price + noise) |
| `return` | Indicates if the order was returned (1 = Yes, 0 = No) |
| `payment_method` | Payment method used |
| `high_value` | Target label: 1 if `purchase_amount` > 200; 0 otherwise |


## Analysis Overview

In this notebook, we will:

1. **Load and explore the data**: Inspect basic characteristics such as column types, missing values, and summary statistics.
2. **Perform exploratory data analysis (EDA)**: Create visualizations to understand distributions and relationships.
3. **Preprocess data**: Encode categorical variables and prepare data for modeling.
4. **Build predictive models**: Train a logistic regression and a random forest classifier to predict whether a purchase is a high-value transaction.
5. **Evaluate models**: Assess model performance using metrics such as accuracy, precision, recall, and F1 score.

Throughout the notebook, we will interpret results and discuss potential business insights.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Load dataset
file_path = 'synthetic_sales_dataset.csv'
df = pd.read_csv(file_path, parse_dates=['purchase_date'])

# Display first few rows
df.head()


In [ ]:

# Basic summary statistics
df.describe(include='all')


In [ ]:

# Plot distribution of purchase amounts
plt.figure(figsize=(8, 5))
sns.histplot(df['purchase_amount'], kde=True, color='skyblue')
plt.title('Distribution of Purchase Amount')
plt.xlabel('Purchase Amount')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Count of purchases by product category
plt.figure(figsize=(8, 5))
sns.countplot(x='product_category', data=df, order=df['product_category'].value_counts().index, palette='pastel')
plt.title('Count of Purchases by Product Category')
plt.xlabel('Product Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Box plot of purchase amount by product category
plt.figure(figsize=(8, 5))
sns.boxplot(x='product_category', y='purchase_amount', data=df, palette='Set2')
plt.title('Purchase Amount by Product Category')
plt.xlabel('Product Category')
plt.ylabel('Purchase Amount')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Correlation heatmap for numerical variables
numeric_cols = ['age', 'units_purchased', 'unit_price', 'purchase_amount', 'return', 'high_value']
plt.figure(figsize=(7, 6))
correlation_matrix = df[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:

# Define features and target
y = df['high_value']
X = df.drop(columns=['high_value', 'purchase_date', 'customer_id'])

# Identify categorical and numerical columns
categorical_cols = ['gender', 'region', 'product_category', 'payment_method']
numerical_cols = ['age', 'units_purchased', 'unit_price', 'purchase_amount', 'return']

# Preprocess data: one-hot encode categorical variables
ohe = OneHotEncoder(drop='first', handle_unknown='ignore')
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', ohe, categorical_cols),
        ('numerical', 'passthrough', numerical_cols)
    ])

# Create train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Define models
log_reg = LogisticRegression(max_iter=1000)
rf = RandomForestClassifier(n_estimators=200, random_state=42)

# Create pipelines
pipe_lr = Pipeline(steps=[('preprocessor', preprocessor), ('model', log_reg)])
pipe_rf = Pipeline(steps=[('preprocessor', preprocessor), ('model', rf)])

# Train models
pipe_lr.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)

# Predict on test set
y_pred_lr = pipe_lr.predict(X_test)
y_pred_rf = pipe_rf.predict(X_test)

# Evaluate models
models = {'Logistic Regression': y_pred_lr, 'Random Forest': y_pred_rf}

for name, y_pred in models.items():
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"
{name} Performance:")
    print(f"Accuracy: {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall: {rec:.3f}")
    print(f"F1 Score: {f1:.3f}")

# Display classification report for the better-performing model
best_model_name = 'Random Forest' if f1_score(y_test, y_pred_rf) > f1_score(y_test, y_pred_lr) else 'Logistic Regression'
print(f"
Best performing model: {best_model_name}
")
if best_model_name == 'Random Forest':
    print(classification_report(y_test, y_pred_rf))
else:
    print(classification_report(y_test, y_pred_lr))


## Conclusion and Next Steps

This analysis showcased how to generate a synthetic e-commerce transaction dataset and perform end-to-end analysis, from exploratory data visualization to predictive modeling. We developed logistic regression and random forest models to classify transactions as high-value or not, demonstrating the ability to handle classification tasks. Future enhancements could include:

- Creating time-series features (e.g., customer purchase history) for improved predictions.
- Developing clustering methods to segment customers for targeted marketing.
- Exploring regression models to forecast sales amounts or customer lifetime value.

This project serves as an excellent starting point for roles such as **Business Analyst**, **Data Analyst**, or **Program Manager**, highlighting skills in data manipulation, visualization, and modeling.
